# Recurrrent Network

## Libraries

In [ ]:
import numpy as np
import tensorflow as tf

from Project5.data import (
    load_split_data,
    create_sequences_1d,
    create_sequences_multifeature,
    split_sequences_by_target_index,
    scale_y_log,
    build_feature_frame,
    scale_multi_features,
)
from Project5.plotting import (
    plan_time_series,
    plan_trend,
    plan_variability,
    plan_possible_seasonality,
    plot_split,
    plot_training_history,
)
from Project5.rnn import build_simple_rnn
from Project5.lstm import build_lstm
from Project5.evaluation import train_model

np.random.seed(42)
tf.random.set_seed(42)

## 1. Data

In [ ]:
HYPERPARAMS = {
    "window": 30,
    "epochs": 100,
    "batch_size": 60,
    "patience": 4,
}

### 1.1. Chronologically splitted + normalized data

In [ ]:
df, (df_train, df_val, df_test), scaler_y = load_split_data("data/amzn_us_d.csv")

print(f"Train: {df_train['Date'].min().date()} -> {df_train['Date'].max().date()}, {len(df_train)} rows")
print(f"Val:   {df_val['Date'].min().date()} -> {df_val['Date'].max().date()}, {len(df_val)} rows")
print(f"Test:  {df_test['Date'].min().date()} -> {df_test['Date'].max().date()}, {len(df_test)} rows")

### 1.2 Preliminary analysis

#### Time Series

In [ ]:
plan_time_series(df)

#### Trend

In [ ]:
plan_trend(df)

#### Variability

In [ ]:
plan_variability(df)

#### Possible Seasonality

In [ ]:
plan_possible_seasonality(df)
plot_split(df_train, df_val, df_test)

### 1.3 Sequence preparation

Apply `log1p` to absorb AMZN's exponential growth, scale with the shared `scaler_y`, then turn the flat series into supervised `(window, 1)` input sequences. The same `train`/`val`/`test` split is preserved by classifying each sequence by the time index of its target.

In [ ]:
y_train_scaled, y_val_scaled, y_test_scaled = scale_y_log(df_train, df_val, df_test, scaler_y)
y_scaled_all = np.concatenate([y_train_scaled, y_val_scaled, y_test_scaled])

WINDOW = HYPERPARAMS["window"]
X_seq, y_seq = create_sequences_1d(y_scaled_all, WINDOW)
target_indices_seq = np.arange(WINDOW, len(y_scaled_all))

train_end_idx = len(y_train_scaled)
val_end_idx = train_end_idx + len(y_val_scaled)

(
    X_train_seq, y_train_seq,
    X_val_seq, y_val_seq,
    X_test_seq, y_test_seq,
    test_mask_seq,
) = split_sequences_by_target_index(X_seq, y_seq, target_indices_seq, train_end_idx, val_end_idx)

print("X_train_seq:", X_train_seq.shape)
print("X_val_seq:  ", X_val_seq.shape)
print("X_test_seq: ", X_test_seq.shape)

## 2. Simple recurrent model RNN

In [ ]:
simple_rnn_model = build_simple_rnn(window=WINDOW)
simple_rnn_model.summary()

history_simple_rnn = train_model(
    simple_rnn_model,
    X_train_seq, y_train_seq,
    X_val_seq, y_val_seq,
    epochs=HYPERPARAMS["epochs"],
    batch_size=HYPERPARAMS["batch_size"],
    patience=HYPERPARAMS["patience"],
)

plot_training_history(history_simple_rnn, "SimpleRNN: training history")

## 3. Recurrent model with memory cells: LSTM

In [ ]:
lstm_model = build_lstm(window=WINDOW, n_features=1)
lstm_model.summary()

history_lstm = train_model(
    lstm_model,
    X_train_seq, y_train_seq,
    X_val_seq, y_val_seq,
    epochs=HYPERPARAMS["epochs"],
    batch_size=HYPERPARAMS["batch_size"],
    patience=HYPERPARAMS["patience"],
)

plot_training_history(history_lstm, "LSTM 1-feature: training history")

## 4. Additional Data

In [ ]:
df_features = build_feature_frame(df)
feature_cols = ["Close", "Open", "High", "Low", "Range"]

n_feat = len(df_features)
train_end_feat = int(0.60 * n_feat)
val_end_feat = int(0.80 * n_feat)

df_feat_train = df_features.iloc[:train_end_feat]
df_feat_val = df_features.iloc[train_end_feat:val_end_feat]
df_feat_test = df_features.iloc[val_end_feat:]

(
    X_train_feat, X_val_feat, X_test_feat,
    y_train_feat, y_val_feat, y_test_feat,
    scaler_X, scaler_y_feat,
) = scale_multi_features(
    df_feat_train, df_feat_val, df_feat_test, df_features,
    feature_cols=feature_cols,
)

X_feat_all = np.concatenate([X_train_feat, X_val_feat, X_test_feat])
y_feat_all = np.concatenate([y_train_feat, y_val_feat, y_test_feat])

WINDOW_FEATURES = HYPERPARAMS["window"]
X_multi, y_multi = create_sequences_multifeature(X_feat_all, y_feat_all, WINDOW_FEATURES)
target_indices_multi = np.arange(WINDOW_FEATURES, len(y_feat_all))

train_end_idx_multi = len(y_train_feat)
val_end_idx_multi = train_end_idx_multi + len(y_val_feat)

(
    X_train_multi, y_train_multi,
    X_val_multi, y_val_multi,
    X_test_multi, y_test_multi,
    test_mask_multi,
) = split_sequences_by_target_index(
    X_multi, y_multi, target_indices_multi, train_end_idx_multi, val_end_idx_multi,
)

print("X_train_multi:", X_train_multi.shape)
print("X_val_multi:  ", X_val_multi.shape)
print("X_test_multi: ", X_test_multi.shape)

multi_lstm_model = build_lstm(window=WINDOW_FEATURES, n_features=len(feature_cols))
multi_lstm_model.summary()

history_multi = train_model(
    multi_lstm_model,
    X_train_multi, y_train_multi,
    X_val_multi, y_val_multi,
    epochs=HYPERPARAMS["epochs"],
    batch_size=HYPERPARAMS["batch_size"],
    patience=HYPERPARAMS["patience"],
)

plot_training_history(history_multi, "LSTM 5-feature: training history")

## 5. Evaluation

## 6. Analysis & Conclusions